# Loading

In [ ]:
from plot_grn import create_network_from_dict,grn_network_from_dict
import omicverse as ov
import pandas as pd
import json
import numpy as np
from adjustText import adjust_text
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
ov.plot_set(font_path='Arial')
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42


In [ ]:
adata = sc.read_h5ad("fetal_brain_data_ChP/hip.h5ad")
import json
with open('Output/GRN_GPU_output/HIP/tmp_files/regulons.json', 'r', encoding='utf-8') as file:
    grn_dict = json.load(file)
adata

# TF expression

In [ ]:
TF_list = []
for TF in grn_dict.keys():
    TF_list.append(TF.split('(')[0])
adata_gene = adata[:,TF_list]
adata_gene

In [ ]:
adata_gene.layers["counts"] = adata_gene.X.copy()
adata_gene.obs_names_make_unique()
sc.pp.normalize_total(adata_gene)
sc.pp.log1p(adata_gene)
#sc.pp.scale(adata_gene)

In [ ]:
sc.tl.dendrogram(adata_gene,'dmt_leiden_anno',use_rep='latent_embeddings_all_spatial_pretrain')
sc.tl.rank_genes_groups(adata_gene, 'dmt_leiden_anno', use_rep='latent_embeddings_all_spatial_pretrain',n_genes=281,
                        method='wilcoxon',use_raw=False,key_added='leiden_wilcoxon')
sc.pl.rank_genes_groups_dotplot(adata_gene,groupby='dmt_leiden_anno',
                                cmap='Spectral_r',key='leiden_wilcoxon',
                                standard_scale='var',n_genes=3,dendrogram=True)

In [ ]:
import numpy as np
import pandas as pd

# 1. 设置要考虑的Top N基因数量
n_top = 3

# 2. 从 AnnData.uns 中获取关键结果
group_names = adata_gene.uns['leiden_wilcoxon']['names'].dtype.names
gene_names_by_group = adata_gene.uns['leiden_wilcoxon']['names']
pvals_adj_by_group = adata_gene.uns['leiden_wilcoxon']['pvals_adj']
scores_by_group = adata_gene.uns['leiden_wilcoxon']['scores'] # 用于次级排序（在p_adj相同或接近时）

# 3. 将所有差异基因结果整理成一个长格式的 DataFrame
# 这种结构便于进行全局排序和去重操作
all_degs = []
for group_name in group_names:
    # 提取当前聚类群的 Top N *所有* 基因（为了有足够的候选基因进行选择）
    # 假设 rank_genes_groups 至少计算了足够多的基因
    
    # 我们只取前 N*K 个基因作为候选池，这里保守地取前10个基因作为候选
    n_candidate = max(n_top, 10) 
    
    current_names = gene_names_by_group[group_name][:n_candidate]
    current_pvals = pvals_adj_by_group[group_name][:n_candidate]
    current_scores = scores_by_group[group_name][:n_candidate]

    df_group = pd.DataFrame({
        'gene': current_names,
        'p_adj': current_pvals,
        'score': current_scores,
        'group': group_name,
        # 添加一个排名，用于后续选择
        'rank_in_group': np.arange(1, len(current_names) + 1)
    })
    all_degs.append(df_group)

# 合并所有数据
full_df = pd.concat(all_degs, ignore_index=True)


# 4. 全局排序：找到每个基因最好的归属细胞类型
# 排序优先级: 1. p_adj (升序, 越小越好) 2. score (降序, 越大越好)
full_df.sort_values(by=['p_adj', 'score'], ascending=[True, False], inplace=True)

# 5. 去重：为每个基因保留 p_adj 最小 (score 最大) 的那一行记录
# 这样，每个基因只被分配给它最佳的细胞类型
unique_best_degs = full_df.drop_duplicates(subset=['gene'], keep='first')


# 6. 为每个细胞类型选择 n_top 个唯一的特异性基因
# 重新初始化结果字典
unique_top_genes_dict = {name: [] for name in group_names}
genes_added_count = {name: 0 for name in group_names}

# 按 p_adj 和 score 再次排序（确保我们优先选择最好的基因）
# 这步排序可以跳过，因为 unique_best_degs 已经基于 p_adj 和 score 排序过了
unique_best_degs.sort_values(by=['p_adj', 'score'], ascending=[True, False], inplace=True)


for index, row in unique_best_degs.iterrows():
    group = row['group']
    gene = row['gene']
    
    # 检查当前细胞类型是否已经收集到 n_top 个基因
    if genes_added_count[group] < n_top:
        unique_top_genes_dict[group].append(gene)
        genes_added_count[group] += 1
        
    if all(count == n_top for count in genes_added_count.values()):
        break

# 7. 打印最终结果
unique_top_genes_dict

In [ ]:
full_df[full_df['group']=='hip_sc_19']

In [ ]:
unique_top_genes_dict = {'ctx_sc_01': ['HMG20B', 'LHX9', 'OTX1'],
 'ctx_sc_02': ['TCF12', 'ZNF704'],
 'ctx_sc_03': [],
 'ctx_sc_04': ['HES6', 'EOMES', 'NHLH1'],
 'ctx_sc_05': ['NFE2', 'CEBPD', 'FOXC1'],
 'ctx_sc_06': ['ZXDA', 'ZNF485', 'CEBPA'],
 'ctx_sc_07': ['NFIB', 'BACH2', 'ATF4'],
 'ctx_sc_08': ['ETV1', 'ZNF467', 'SOX15'],
 'ctx_sc_09': ['SOX11', 'BCL11A', 'HDAC2'],
 'ctx_sc_10': ['MAZ', 'MAF'],
 'ctx_sc_11': ['ZEB1', 'FOXF2', 'ETS1'],
 'ctx_sc_12': ['PBX1', 'HIVEP2', 'NR2C2'],
 'ctx_sc_13': ['RFX3', 'ZNF148', 'HLTF'],
 'ctx_sc_14': ['NFE2L2', 'SOX21'],
 'ctx_sc_15': ['HMGA2'],
 'ctx_sc_16': ['SOX9', 'POU3F3', 'JUND'],
 'ctx_sc_17': ['ERF'],
 'hip_sc_18': ['KLF7','THRA'],
 'hip_sc_19': ['FOSL2'],
 'hip_sc_20': ['SREBF2'],
 'hip_sc_21': ['NFIC', 'TCF7L2'],
 'hip_sc_22': ['TCF4', 'NFIA', 'NEUROD1'],
 'hip_sc_23': ['EMX2', 'LEF1']}

In [ ]:
ax = sc.pl.matrixplot(
    adata_gene,
    unique_top_genes_dict,
    "dmt_leiden_anno",
    dendrogram=True,
    colorbar_title="mean expression",
    standard_scale = 'var',
    vmin=0,
    vmax=1,
    cmap="GnBu",
    show=False,
    figsize=(18,6),
)
plt.savefig("Figure/heatmap/hip_tf.pdf", dpi=300, bbox_inches = "tight")

In [ ]:
unique_top_genes_dict = {
 'hip_sc_18': ['KLF7','THRA'],
 'hip_sc_19': ['FOSL2'],
 'hip_sc_20': ['SREBF2'],
 'hip_sc_21': ['NFIC', 'TCF7L2'],
 'hip_sc_22': ['TCF4', 'NFIA', 'NEUROD1'],
 'hip_sc_23': ['EMX2', 'LEF1']}

In [ ]:
adata_gene_part = adata_gene[adata_gene.obs['dmt_leiden_anno'].isin(['hip_sc_18','hip_sc_19','hip_sc_20','hip_sc_21','hip_sc_22','hip_sc_23'])]
adata_gene_part.obs['dmt_leiden_anno'] = adata_gene_part.obs['dmt_leiden_anno'].astype("str")
adata_gene_part.obs['dmt_leiden_anno'] = adata_gene_part.obs['dmt_leiden_anno'].astype("category")
sc.tl.dendrogram(adata_gene_part,groupby='dmt_leiden_anno')
ax = sc.pl.matrixplot(
    adata_gene_part,
    unique_top_genes_dict,
    "dmt_leiden_anno",
    dendrogram=True,
    colorbar_title="mean expression",
    standard_scale = 'var',
    vmin=0,
    vmax=1,
    cmap="GnBu",
    show=False,
    figsize=(7,2.5),
)
plt.savefig("Figure/heatmap/hip_tf_subcluster.pdf", dpi=300, bbox_inches = "tight")

# GRN score

In [ ]:
auc_mtx = sc.read_h5ad("Process_Data/specific_region/auc_hip.h5ad")

In [ ]:
def add_suffix_to_dict_list(input_dict, suffix='(+)'):
    return {k: [f"{e}{suffix}" for e in v] for k, v in input_dict.items()}
unique_top_genes_dict = add_suffix_to_dict_list(unique_top_genes_dict)
unique_top_genes_dict

In [ ]:
auc_mtx.uns = adata_gene.uns
ax = sc.pl.matrixplot(
    auc_mtx,
    unique_top_genes_dict,
    "dmt_leiden_anno",
    dendrogram=True,
    colorbar_title="mean AUCell score",
    standard_scale = 'var',
    vmin=0,
    vmax=1,
    cmap="OrRd",
    show=False,
    figsize=(18,6),
)
plt.savefig("Figure/heatmap/hip_grn.pdf", dpi=300, bbox_inches = "tight")